# 🤖 LangGraph Multi-Agent Research Demo — Colab 一鍵跑

> **目標**:30 分鐘內跑通一個 3-agent supervisor 系統,輸出結構化研究報告
>
> **架構**:Supervisor → Planner / Researcher / Writer(+ Tavily web search tool)
>
> **模型**:GPT-4o-mini(便宜、快、夠用);Claude / Gemini 可一行替換
>
> **預期成本**:單次 demo run 約 $0.03-0.10(視 query 複雜度)
>
> **不需 GPU** — 全部用 API,Colab CPU 即可

---

## 對應的 deep-dive 與 case study

- 概念解析:[`../LangGraph_supervisor_handoff_實戰.md`](../LangGraph_supervisor_handoff_實戰.md)
- 系統設計案例:[Case_04 Multi-Agent Research System](../../../9.面試準備與職業發展/2.系統設計案例/Case_04_Multi_Agent_Research_System.md)
- 框架對比:[`../Agent框架選擇決策指南.md`](../Agent框架選擇決策指南.md)

## phantom-mesh 寫由

在 phantom-mesh 中,Supervisor + Handoff 是 multi-agent 協調的標準模式:
1. **provider abstraction**:Researcher 用 OpenAI、Writer 換 Anthropic 也行(本 notebook 一行可換)
2. **streaming SSE 統一解析**:每個 agent 的輸出都 stream 給 supervisor,phantom-mesh 的 stream parser 對齊各家
3. **cost tracking**:per-task / per-agent / per-step 三層 attribution
4. **failure recovery**:checkpoint 在每個 handoff 後寫,中途失敗可 resume

---

## 0️⃣ 環境準備

需要兩把 API key:
1. **OpenAI**(用 GPT-4o-mini,免費 trial 額度即可)— https://platform.openai.com/api-keys
2. **Tavily**(免費 1000 search/月)— https://app.tavily.com/home

替換成 Claude / Gemini 也可,本 notebook 末段有範例。

In [ ]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')
if not os.environ.get('TAVILY_API_KEY'):
    os.environ['TAVILY_API_KEY'] = getpass.getpass('Tavily API key: ')

print('✅ API keys 已設定')

## 1️⃣ 安裝套件

用 2026-05 兼容版本。

In [ ]:
%%capture
!pip install -U "langgraph>=0.2.50" "langchain>=0.3" "langchain-openai>=0.2" "langchain-community>=0.3" "tavily-python>=0.5" "pydantic>=2" rich

In [ ]:
from typing import Annotated, Literal, TypedDict, List
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from pydantic import BaseModel, Field
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

console = Console()
print('✅ Imports OK')

## 2️⃣ 定義 State

LangGraph 的 State 是「節點間共享的記憶體」。我們用 TypedDict 定義,讓型別清晰:

- `messages`:所有對話歷史(supervisor 看的)
- `plan`:planner 拆出來的 sub-questions
- `findings`:researcher 找到的事實
- `report`:writer 整合的最終報告
- `next_agent`:supervisor 決定下一步給誰

In [ ]:
class ResearchState(TypedDict):
    messages: Annotated[list, add_messages]
    query: str
    plan: List[str]
    findings: List[dict]  # [{'question': ..., 'answer': ..., 'sources': [...]}]
    report: str
    next_agent: Literal['planner', 'researcher', 'writer', 'END']
    iteration: int  # cost guard

## 3️⃣ 定義 Tools

Researcher 用 Tavily 搜尋。可以加更多 tool(arXiv、Wikipedia、code exec)— 這份 demo 只用一個保持簡潔。

In [ ]:
search_tool = TavilySearchResults(max_results=4, search_depth='advanced')

# 測試一下
test_result = search_tool.invoke('GraphRAG benchmark 2025')
print(f'✅ Tavily 可用,首筆結果 url:{test_result[0]["url"] if test_result else "(無結果)"}')

## 4️⃣ 定義 LLM

Supervisor / Planner / Writer 用 GPT-4o-mini;若要更聰明可換 GPT-4o 或 Claude。

In [ ]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.2)
# 替代:llm = ChatOpenAI(model='gpt-4o', temperature=0.2)
# 替代:from langchain_anthropic import ChatAnthropic; llm = ChatAnthropic(model='claude-3-5-haiku-20241022')

print(f'LLM:{llm.model_name}')

## 5️⃣ Planner Agent

把 user query 拆成 3-5 個 sub-question,寫進 `state['plan']`。

**phantom-mesh 對應**:planner 是 cost-aware 的關鍵 — 拆得太多 sub-question 會 token 爆炸,本範例用 structured output 限制 4 條。

In [ ]:
class Plan(BaseModel):
    sub_questions: List[str] = Field(
        description='3-5 個具體、可獨立檢索的 sub-questions',
        max_length=5, min_length=3,
    )

PLANNER_PROMPT = """你是一個研究 planner。
User 問:「{query}」

請拆成 3-5 個具體、可獨立檢索的 sub-questions。每個 sub-question 應該:
1. 可以單獨用 web search 找到答案
2. 彼此互補但不重複
3. 涵蓋:定義/原理、現況、對比/trade-off、未來趨勢

用繁體中文。"""

def planner_node(state: ResearchState) -> dict:
    structured_llm = llm.with_structured_output(Plan)
    result = structured_llm.invoke(PLANNER_PROMPT.format(query=state['query']))
    console.print(Panel(
        '\n'.join(f'{i+1}. {q}' for i, q in enumerate(result.sub_questions)),
        title='📋 Planner — sub-questions', border_style='cyan',
    ))
    return {
        'plan': result.sub_questions,
        'messages': [AIMessage(content=f'Plan:\n' + '\n'.join(result.sub_questions))],
    }

## 6️⃣ Researcher Agent

對每個 sub-question 跑 Tavily 搜尋 → 用 LLM 摘要 → 寫進 `state['findings']`。

**重點**:這裡可以 parallel(同時跑 4 個 sub-question),但為了 demo 簡潔用 sequential。實戰用 `langgraph` 的 `Send` 操作可做 fan-out 並行。

In [ ]:
RESEARCH_PROMPT = """你是研究員。
Sub-question:{question}

以下是 web 搜尋結果:
{search_results}

請寫一段 80-150 字的繁體中文摘要回答 sub-question。
**必須包含**:
1. 明確答案
2. 至少 2 個關鍵數字或事實
3. 在末尾用方括號標註來源 URL,如 [https://...]
"""

def researcher_node(state: ResearchState) -> dict:
    findings = []
    for q in state['plan']:
        search_results = search_tool.invoke(q)
        formatted = '\n\n'.join([f"來源: {r['url']}\n{r['content']}" for r in search_results[:3]])
        answer = llm.invoke(RESEARCH_PROMPT.format(question=q, search_results=formatted)).content
        findings.append({
            'question': q,
            'answer': answer,
            'sources': [r['url'] for r in search_results[:3]],
        })
        console.print(Panel(
            f'[bold]{q}[/bold]\n\n{answer[:300]}{"..." if len(answer) > 300 else ""}',
            title=f'🔍 Research', border_style='yellow',
        ))
    return {'findings': findings}

## 7️⃣ Writer Agent

把所有 findings 整合成結構化 Markdown 報告。Writer 不重新做搜尋,只做整合與排版。

In [ ]:
WRITER_PROMPT = """你是技術 writer。請整合以下研究結果寫成結構化的 Markdown 報告。

User 原始問題:「{query}」

研究結果:
{findings}

報告格式:
# 標題(從 query 提煉)

## 摘要
3-5 句總結整篇

## 主要發現
用 H3 章節組織,每節對應一個 sub-question。
**保留所有來源 URL 引用(用方括號)**。

## 結論與展望
2-3 段總結 + 對未來的看法

## 參考來源
列出所有 URL(去重)

用繁體中文,專業但易讀。"""

def writer_node(state: ResearchState) -> dict:
    findings_text = '\n\n'.join([
        f'### {f["question"]}\n{f["answer"]}\n來源:{", ".join(f["sources"])}'
        for f in state['findings']
    ])
    report = llm.invoke(WRITER_PROMPT.format(
        query=state['query'], findings=findings_text
    )).content
    return {'report': report, 'next_agent': 'END'}

## 8️⃣ Supervisor — 路由邏輯

Supervisor 不調用 LLM,只看 state 決定下一步給誰:
- 沒 plan → 給 Planner
- 有 plan 沒 findings → 給 Researcher
- 有 findings 沒 report → 給 Writer
- 有 report 或超過 iteration 上限 → END

**Cost guard**:`iteration > 5` 強制 END,避免無窮 loop 燒 token。實戰可用 token / $ budget 替換。

In [ ]:
def supervisor(state: ResearchState) -> dict:
    iteration = state.get('iteration', 0) + 1
    if iteration > 5:
        return {'next_agent': 'END', 'iteration': iteration}
    if not state.get('plan'):
        return {'next_agent': 'planner', 'iteration': iteration}
    if not state.get('findings'):
        return {'next_agent': 'researcher', 'iteration': iteration}
    if not state.get('report'):
        return {'next_agent': 'writer', 'iteration': iteration}
    return {'next_agent': 'END', 'iteration': iteration}

def route(state: ResearchState) -> str:
    return state['next_agent']

## 9️⃣ 編譯 Graph

把節點與 conditional edge 接起來,加 MemorySaver 做 checkpoint(production 可換 SqliteSaver / PostgresSaver)。

In [ ]:
graph = StateGraph(ResearchState)
graph.add_node('supervisor', supervisor)
graph.add_node('planner', planner_node)
graph.add_node('researcher', researcher_node)
graph.add_node('writer', writer_node)

graph.add_edge(START, 'supervisor')
graph.add_conditional_edges('supervisor', route, {
    'planner': 'planner',
    'researcher': 'researcher',
    'writer': 'writer',
    'END': END,
})
graph.add_edge('planner', 'supervisor')
graph.add_edge('researcher', 'supervisor')
graph.add_edge('writer', 'supervisor')

app = graph.compile(checkpointer=MemorySaver())
print('✅ Graph 編譯完成')
print('架構: START → supervisor ↔ {planner, researcher, writer} → END')

## 🔟 可視化 Graph(可選)

如果你的環境有 graphviz / mermaid 支援,可以畫出 graph 拓樸。

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f'(畫圖失敗,環境不支援:{type(e).__name__})')
    print('Mermaid 文字形式:')
    print(app.get_graph().draw_mermaid())

## 1️⃣1️⃣ 跑 Demo Query

選一個 2026 frontier 主題試試。預期執行 4 步(Planner → Researcher × 4 sub-questions → Writer)、總 token ~20-40K。

In [ ]:
USER_QUERY = '2025 GraphRAG 跟 vector RAG 比有什麼優勢和缺點?有什麼真實 production 案例?'

config = {'configurable': {'thread_id': 'demo-1'}}
result = app.invoke({
    'query': USER_QUERY,
    'messages': [HumanMessage(content=USER_QUERY)],
    'plan': [],
    'findings': [],
    'report': '',
    'next_agent': 'planner',
    'iteration': 0,
}, config=config)

print(f'\n✅ 完成,總 iteration: {result["iteration"]}')
print(f'   plan 條數: {len(result["plan"])}')
print(f'   findings 條數: {len(result["findings"])}')
print(f'   report 長度: {len(result["report"])} 字元')

## 1️⃣2️⃣ 顯示最終報告

In [ ]:
console.print(Panel(Markdown(result['report']), title='📄 Final Report', border_style='green'))

## 1️⃣3️⃣ Checkpoint 與 Resume

LangGraph 在每個 super-step 後自動 checkpoint。我們可以列出歷史 state,或從中段繼續。

In [ ]:
# 列出當前 thread 的所有 checkpoint
history = list(app.get_state_history(config))
print(f'本次 run 共 {len(history)} 個 checkpoint')
print(f'\n最後三個 checkpoint:')
for ckpt in history[:3]:
    print(f'  step {ckpt.metadata["step"]}: next_agent={ckpt.values.get("next_agent")}, findings={len(ckpt.values.get("findings", []))}')

## 1️⃣4️⃣ phantom-mesh 真實工程考量

本 demo 用 Memory checkpointer,實戰需考慮:

### 14.1 Checkpoint persistence
- Production:`SqliteSaver` / `PostgresSaver` 或 Redis,確保程式 crash 後可 resume
- 對應 [Case_04 §5.7 Memory & State](../../../9.面試準備與職業發展/2.系統設計案例/Case_04_Multi_Agent_Research_System.md)

### 14.2 Parallel fan-out
- 本 demo 的 Researcher 是 sequential。生產用 `Send` 操作改 parallel:`return [Send('researcher', {'question': q}) for q in state['plan']]`
- 速度從 4× search latency → 1× search latency

### 14.3 Cost & Token Tracking
- 本 demo 沒精細追蹤。生產接 Langfuse / LangSmith,每個 node 自動 trace token + $
- 設 per-task budget cap,超過自動降級到便宜模型
- 對應 [Case_02 §5.5 Cost Tracker](../../../9.面試準備與職業發展/2.系統設計案例/Case_02_LLM_Gateway_API_Platform.md)

### 14.4 Provider Fallback
- GPT-4o-mini 掛掉 → 自動切 Claude Haiku
- 用 LangChain 的 `with_fallbacks`:`llm.with_fallbacks([claude_llm, gemini_llm])`
- phantom-mesh provider abstraction 的核心

### 14.5 Streaming SSE
- 把 graph 改成 `app.stream(...)` 即可 token-by-token 推送給前端
- 每個 agent 的中間結果都即時可見(進度條 UX)
- 對應 [Case_03 Voice Agent](../../../9.面試準備與職業發展/2.系統設計案例/Case_03_Voice_Agent_Customer_Service.md) 的 streaming 設計

### 14.6 HITL (Human-in-the-Loop)
- 在 supervisor 後 / writer 前加 `interrupt`,user 可改 plan 或要求重做
- 對應 [`../LangGraph_supervisor_handoff_實戰.md`](../LangGraph_supervisor_handoff_實戰.md) §HITL

### 14.7 Tool Allowlist & Sandbox
- Tavily 是 read-only,安全;若加 code exec 必須 sandbox
- 對應 [Case_05 Computer Use SaaS](../../../9.面試準備與職業發展/2.系統設計案例/Case_05_Computer_Use_SaaS.md) 的 prompt injection 防禦

---

## 🔬 擴展練習

1. **加 Critic Agent**:在 Writer 前插入 Critic node,檢查 findings 是否完整、有沒有矛盾,不通過則 re-research
2. **Parallel Researcher fan-out**:用 `Send` 把 4 個 sub-question 同時跑,latency 砍 4 倍
3. **加更多 tool**:arXiv search、Wikipedia、code interpreter
4. **Fact-Checker**:對 writer 寫出來的每個事實句,跑一輪 verify(回查 source)
5. **HITL**:在 Planner 之後加 `interrupt()`,讓 user 確認 sub-questions 才繼續
6. **接 MCP**:把 search tool 用 MCP server 包裝,跨 agent 統一 tool 介面 — 見 [`../../11.MCP協議與工具調用/MCP_server_完整開發.md`](../../11.MCP協議與工具調用/MCP_server_完整開發.md)
7. **接 Mem0 / Zep**:把 findings 寫入長期記憶,跨 session reuse
8. **加 SWE-Bench 風 eval**:設計 30 個 ground truth research query 跑分,觀察修改後品質變化

---

## 📚 References

- [LangGraph 官方文檔](https://langchain-ai.github.io/langgraph/)
- [LangGraph Multi-Agent Tutorial](https://langchain-ai.github.io/langgraph/tutorials/multi_agent/agent_supervisor/)
- [Anthropic - Building Effective Agents](https://www.anthropic.com/research/building-effective-agents)
- 本 repo:[`../LangGraph_supervisor_handoff_實戰.md`](../LangGraph_supervisor_handoff_實戰.md)、[`../Claude_Agent_SDK_DeepAgents.md`](../Claude_Agent_SDK_DeepAgents.md)

---

**Last updated**: 2026-05-16  
**Tested on**: Colab CPU(無 GPU 需求),Python 3.10,langgraph 0.2.50,langchain 0.3